# Gemma stem paraphraser — AyurShloka-QA v2The 80k question set is generated by deterministic templates, so every item of a givenkind opens the same way ("Fill in the blank in this verse from…"). A model fine-tuned onthat can learn the **template** rather than the content.This notebook has Gemma rewrite the **question stem only**. Options, answer key, answertext and all provenance are copied through untouched — so the paraphrase cannot changewhat the correct answer is, and a bad generation can only ever cost you variety, nevercorrectness.**What it does not do:** it does not invent questions, and it does not touch the answer.Gemma never sees the answer key.| Cell | What to change ||---|---|| **Section 3 — CONFIG** | `MODEL_NAME`, `LIMIT`, `BATCH_SIZE` || **Section 4** | `DATA_PATH` only if auto-detect fails |**Before running:** Settings → Accelerator → **GPU T4 x2**, Internet → **On**.Gemma is gated: accept the licence on its HuggingFace page and add an `HF_TOKEN` secret.**Runtime:** roughly 2–4 GPU hours for all 80k at batch 32 on a T4. Checkpoints every2,000 rows, so you can stop and resume across Kaggle sessions.

## 1. Install

In [ ]:
!pip install -q -U transformers accelerate bitsandbytesimport torch, transformersprint("transformers", transformers.__version__)print("CUDA:", torch.cuda.is_available(),      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. HF tokenGemma is gated. Accept the licence at huggingface.co/google/gemma-2-2b-it, then**Add-ons → Secrets** with key `HF_TOKEN`.

In [ ]:
import ostry:    from kaggle_secrets import UserSecretsClient    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")    print("HF_TOKEN loaded")except Exception as e:    print("No HF_TOKEN:", e)

## 3. 🔧 CONFIG`gemma-2-2b-it` is the sweet spot on a T4 — big enough to paraphrase cleanly, smallenough to run the whole set in a few hours. `gemma-2-9b-it` needs `LOAD_IN_4BIT = True`.Set `LIMIT` to a few hundred for a trial run before committing to all 80k.

In [ ]:
MODEL_NAME   = "google/gemma-2-2b-it"LIMIT        = 500          # 0 = all 80,000BATCH_SIZE   = 32           # drop to 16 if you hit OOMMAX_NEW      = 96LOAD_IN_4BIT = False        # True for gemma-2-9b-it on a single T4CHECKPOINT_EVERY = 2000OUT_PATH     = "/kaggle/working/ayur_qa_v2_80k_paraphrased.jsonl"

## 4. Locate the datasetAttach the `ayur_qa_v2` files as a Kaggle dataset (`ayur_qa_v2_80k.jsonl`, or the`train.jsonl` / `validation.jsonl` splits).

In [ ]:
from pathlib import Pathimport jsonDATA_PATH = None   # <-- set explicitly to override auto-detectif DATA_PATH is None:    hits = sorted(Path("/kaggle/input").rglob("ayur_qa_v2_80k.jsonl")) \        if Path("/kaggle/input").exists() else []    if not hits:        hits = sorted(Path("/kaggle/input").rglob("*.jsonl")) \            if Path("/kaggle/input").exists() else []    DATA_PATH = hits[0] if hits else Noneprint("DATA_PATH:", DATA_PATH)rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8") if l.strip()]print("rows:", len(rows))print("types:", {t: sum(1 for r in rows if r["question_type"] == t)                 for t in sorted({r["question_type"] for r in rows})})

## 5. What may be rewrittenSome stems must survive verbatim or the item breaks:* **Fill in the blanks** — the `________` marker must stay, and the quoted verse around it* **Match the column** — the Column A / Column B block is structured, not prose* **Assertion or Reasoning** — the `Assertion (A):` / `Reason (R):` statements are the itemFor those, only the *instruction wrapper* is safe to touch, so the notebook skips them bydefault. MCQ, Multiple selection and Assorted are paraphrased.

In [ ]:
PARAPHRASABLE = {"MCQ", "Multiple selection", "Assorted"}# never paraphrase a stem that carries a verse or a structural markerimport redef is_safe(r):    if r["question_type"] not in PARAPHRASABLE:        return False    q = r["question"]    if "____" in q or "Column A" in q or "Assertion (A):" in q:        return False    if re.search(r"[\u0900-\u097F]", q):     # Devanagari in the stem - keep verbatim        return False    return len(q) < 900todo = [r for r in rows if is_safe(r)]print("paraphrasable: %d of %d (%.0f%%)" % (len(todo), len(rows), 100*len(todo)/len(rows)))print("skipped by type:", {t: sum(1 for r in rows if r["question_type"] == t and not is_safe(r))                           for t in sorted({r["question_type"] for r in rows})})

## 6. Load Gemma

In [ ]:
import torchfrom transformers import AutoTokenizer, AutoModelForCausalLMtok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)if tok.pad_token is None:    tok.pad_token = tok.eos_tokentok.padding_side = "left"          # decoder-only + batched generationkw = dict(device_map={"": 0}, trust_remote_code=True)if LOAD_IN_4BIT:    from transformers import BitsAndBytesConfig    kw["quantization_config"] = BitsAndBytesConfig(        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")else:    kw["dtype"] = torch.float16try:    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kw)except TypeError:    kw["torch_dtype"] = kw.pop("dtype", torch.float16)    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kw)model.eval()print("params: %.2fB" % (sum(p.numel() for p in model.parameters())/1e9))

## 7. Prompt and guardsThe prompt asks only for a rephrasing. The guard then rejects any output that drifts —too short, too long, lost the subject, or started answering the question — and falls backto the original stem. A rejected paraphrase costs variety, never correctness.

In [ ]:
INSTR = ( "Rewrite the following exam question so it asks exactly the same thing in different words.\n" "Rules:\n" "- Keep every Sanskrit term, proper noun, number and citation exactly as written.\n" "- Do NOT answer the question.\n" "- Do NOT add or remove information.\n" "- Do NOT mention options A, B, C or D.\n" "- Return ONLY the rewritten question, nothing else.\n\n" "Question: {q}\n\nRewritten question:")def build(q):    msgs = [{"role": "user", "content": INSTR.format(q=q)}]    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)KEY_TOKENS = re.compile(r"[A-Za-zĀĪŪṚṜḶṄÑṬḌṆŚṢṀḤāīūṛṝḷṅñṭḍṇśṣṃḥ]{5,}|\d+")def accept(orig, new):    """Reject drift; the caller then keeps the original stem."""    if not new or len(new) < 20:        return False    if not (0.5 * len(orig) <= len(new) <= 1.8 * len(orig) + 60):        return False    low = new.lower()    if low.startswith(("answer", "the answer", "sure", "here is", "here's", "rewritten")):        return False    if re.search(r"\boption [a-e]\b|\([a-e]\)", low):        return False    # the rewrite must retain most of the original's content words / numbers    o = set(KEY_TOKENS.findall(orig.lower()))    n = set(KEY_TOKENS.findall(low))    return not o or len(o & n) / len(o) >= 0.55@torch.no_grad()def paraphrase(batch):    enc = tok([build(q) for q in batch], return_tensors="pt",              padding=True, truncation=True, max_length=1024).to(model.device)    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=True,                         temperature=0.8, top_p=0.9, pad_token_id=tok.pad_token_id)    res = []    for i in range(out.shape[0]):        g = out[i][enc["input_ids"].shape[1]:]        res.append(tok.decode(g, skip_special_tokens=True).strip().strip('"'))    return res

## 8. Smoke test — read these before running the full passIf the accept-rate here is under ~60%, the prompt or the model is not working for yoursetup; fix that before spending hours.

In [ ]:
sample = todo[:8]outs = paraphrase([r["question"] for r in sample])ok = 0for r, o in zip(sample, outs):    good = accept(r["question"], o)    ok += good    print("=" * 90)    print("[%s] %s" % ("ACCEPT" if good else "REJECT", r["question_type"]))    print("  original : %s" % r["question"][:220].replace("\n", " "))    print("  rewritten: %s" % o[:220].replace("\n", " "))print("\naccept rate: %d/%d" % (ok, len(sample)))

## 9. RunWrites `question_original` alongside the rewritten `question`, plus a`paraphrased` flag, so you can always recover the deterministic stem or measure how muchvariety you actually gained.

In [ ]:
import time, jsonwork = todo if not LIMIT else todo[:LIMIT]by_id = {id(r): r for r in work}done, kept, t0 = 0, 0, time.time()results = {}for i in range(0, len(work), BATCH_SIZE):    chunk = work[i:i + BATCH_SIZE]    try:        outs = paraphrase([r["question"] for r in chunk])    except torch.cuda.OutOfMemoryError:        torch.cuda.empty_cache()        print("OOM at row %d - lower BATCH_SIZE and re-run" % i)        break    for r, o in zip(chunk, outs):        if accept(r["question"], o):            results[r["question_id"]] = o            kept += 1    done += len(chunk)    if done % CHECKPOINT_EVERY < BATCH_SIZE:        rate = done / max(time.time() - t0, 1e-9)        print("%6d/%d  accepted %5d (%.0f%%)  %.1f rows/s  eta %.1f min"              % (done, len(work), kept, 100*kept/max(done,1), rate,                 (len(work)-done)/max(rate,1e-9)/60))        json.dump(results, open("/kaggle/working/_paraphrase_ckpt.json", "w"), ensure_ascii=False)json.dump(results, open("/kaggle/working/_paraphrase_ckpt.json", "w"), ensure_ascii=False)print("\ndone: %d rewritten of %d attempted (%.0f%%) in %.1f min"      % (kept, done, 100*kept/max(done,1), (time.time()-t0)/60))

## 10. Write the paraphrased dataset

In [ ]:
n = 0with open(OUT_PATH, "w", encoding="utf-8") as fh:    for r in rows:        new = results.get(r["question_id"])        if new:            r = dict(r)            r["question_original"] = r["question"]            r["question"] = new            r["paraphrased"] = True            n += 1        else:            r = dict(r)            r["paraphrased"] = False        fh.write(json.dumps(r, ensure_ascii=False) + "\n")print("wrote %s  (%d of %d rows paraphrased)" % (OUT_PATH, n, len(rows)))

## 11. Verify nothing but the stem movedThis is the check that matters. If any answer key, option or provenance field changed,the run is not safe to train on.

In [ ]:
src = {r["question_id"]: r for r in rows}out = [json.loads(l) for l in open(OUT_PATH, encoding="utf-8") if l.strip()]FROZEN = ["correct_answer", "answer_text", "question_type", "fact_id", "chunk_id",          "source_book", "option_a", "option_b", "option_c", "option_d", "option_e"]bad = 0for r in out:    o = src[r["question_id"]]    for f in FROZEN:        if str(o.get(f, "")) != str(r.get(f, "")):            bad += 1            print("CHANGED %s.%s" % (r["question_id"], f))            breakstems = [r["question"] for r in out]print("rows            :", len(out))print("frozen-field drift:", bad, "(must be 0)")print("unique stems    :", len(set(stems)), "(duplicates:", len(stems)-len(set(stems)), ")")print("paraphrased     :", sum(1 for r in out if r.get("paraphrased")))assert bad == 0, "answers or options changed - do NOT use this output"print("\nSAFE: only question stems changed.")